<a href="https://colab.research.google.com/github/kayokfds/trabalhoooo/blob/main/03_Base_Fatores_Macro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Dados estão no Google Drive, em FGV > TCC > Dados

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount("/content/drive")
import os
import requests
import zipfile
from io import BytesIO
from tqdm import tqdm
from datetime import datetime, timedelta

Mounted at /content/drive


In [ ]:
caminho_dados = "/content/drive/MyDrive/FGV/TCC/Dados/Brutos"
caminho_output = "/content/drive/MyDrive/FGV/TCC/Dados"

## Selic

In [ ]:
dados_selic = pd.read_csv(f"{caminho_dados}/Selic/selic.csv", sep=";", decimal=',')

In [ ]:
df_selic = dados_selic.copy()
print(df_selic.shape)
df_selic.columns = ['data_referencia','selic']
df_selic = df_selic[~(df_selic['data_referencia'] == 'Fonte')]
df_selic['data_referencia'] = pd.to_datetime(df_selic['data_referencia'], format = '%d/%m/%Y')
df_selic['selic'] = pd.to_numeric(df_selic['selic'].str.replace(',','.'))
print(df_selic.dtypes)
df_selic

(6687, 2)
data_referencia    datetime64[ns]
selic                     float64
dtype: object


,data_referencia,selic
0,2000-01-03,19.04
1,2000-01-04,19.04
2,2000-01-05,19.05
3,2000-01-06,19.07
4,2000-01-07,19.07
...,...,...
6681,2026-08-10,13.90
6682,2026-08-11,13.90
6683,2026-08-12,13.90
6684,2026-08-13,13.90


##Spread

In [ ]:
dados_spread = pd.read_csv(f"{caminho_output}/dados_term_spread.csv")
dados_spread_2 = pd.read_csv(f"{caminho_output}/dados_term_spread_complementar.csv")

In [ ]:
df_spread = pd.concat([dados_spread,dados_spread_2])

alvos_dc = [3650, 730, 365, 182, 91]  # equivalentes em dias corridos ~10Y 2Y 1Y 6M 3M
nomes_colunas = ["2520_du", "504_du", "252_du", "126_du", "63_du"]

linhas_saida = []

for data, grupo in df_spread.groupby("data"):
    grupo = grupo.sort_values("dias_corridos")
    dcs = grupo["dias_corridos"].values
    taxas = grupo["taxa"].values

    if len(dcs) < 2:
        continue

    taxas_interpoladas = np.interp(alvos_dc, dcs, taxas)

    linha = {"data_referencia": data}
    for nome, valor in zip(nomes_colunas, taxas_interpoladas):
        linha[nome] = valor
    linhas_saida.append(linha)

df_spread = pd.DataFrame(linhas_saida)
df_spread['data_referencia'] = pd.to_datetime(df_spread['data_referencia'])
df_spread.head(10)

,data_referencia,2520_du,504_du,252_du,126_du,63_du
0,2016-01-04,16.420130,16.501154,15.763429,15.224000,14.750000
1,2016-01-05,16.070000,16.212000,15.606000,15.114182,14.665333
2,2016-01-06,15.960085,16.073462,15.529000,15.072545,14.669250
3,2016-01-07,16.150170,16.154615,15.552571,15.066000,14.670714
4,2016-01-08,16.230255,16.155769,15.547143,15.104778,14.700167
5,2016-01-11,16.308383,16.209231,15.628545,15.163000,14.738000
6,2016-01-12,16.297298,16.161077,15.554000,15.112250,14.725000
7,2016-01-13,16.251532,16.152308,15.497429,15.094000,14.735000
8,2016-01-14,16.381340,16.255333,15.520857,15.106000,14.748500
9,2016-01-15,16.471489,16.331364,15.587143,15.139000,14.780000


In [ ]:
# df_spread.to_csv(f"{caminho_output}/output_term_spread.csv", index=False)
# print(f"Fim. {len(df_spread)} dias salvos em {caminho_output}/output_term_spread.csv")

## Câmbio

In [ ]:
dados_cambio = pd.read_json(f"{caminho_dados}/Câmbio/cambio.txt")
dados_cambio.head()

,data,valor
0,04/01/2016,4.0387
1,05/01/2016,4.0114
2,06/01/2016,4.0303
3,07/01/2016,4.0475
4,08/01/2016,4.0250


In [ ]:
df_cambio = dados_cambio.copy()
df_cambio = df_cambio.rename(columns={'data': 'data_referencia', 'valor': 'cambio'})
df_cambio['data_referencia'] = pd.to_datetime(df_cambio['data_referencia'], format='%d/%m/%Y')
print(df_cambio.shape)
print(df_cambio.dtypes)
df_cambio

(2510, 2)
data_referencia    datetime64[ns]
cambio                    float64
dtype: object


,data_referencia,cambio
0,2016-01-04,4.0387
1,2016-01-05,4.0114
2,2016-01-06,4.0303
3,2016-01-07,4.0475
4,2016-01-08,4.0250
...,...,...
2505,2025-12-24,5.5350
2506,2025-12-26,5.5413
2507,2025-12-29,5.5739
2508,2025-12-30,5.5024


## Juntando/Exportando

In [ ]:
df_final = pd.merge(df_selic, df_spread, on='data_referencia', how='outer')
df_final = pd.merge(df_final, df_cambio, on='data_referencia', how='outer')
print(df_final.dtypes)
df_final.head()

data_referencia    datetime64[ns]
selic                     float64
2520_du                   float64
504_du                    float64
252_du                    float64
126_du                    float64
63_du                     float64
cambio                    float64
dtype: object


,data_referencia,selic,2520_du,504_du,252_du,126_du,63_du,cambio
0,2000-01-03,19.04,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-01-04,19.04,NaN,NaN,NaN,NaN,NaN,NaN
2,2000-01-05,19.05,NaN,NaN,NaN,NaN,NaN,NaN
3,2000-01-06,19.07,NaN,NaN,NaN,NaN,NaN,NaN
4,2000-01-07,19.07,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_final.to_csv(f"{caminho_output}/dados_macro.csv")